## Merge AIIE → crosswalk → CPS

# AIIE

We create and validate a new measure of an occupation's exposure to AI that we call the AI Occupational Exposure (AIOE). We use the AIOE to construct a measure of AI exposure at the industry level, which we call the AI Industry Exposure (AIIE) and a measure of AI exposure at the county level, which we call the AI Geographic Exposure (AIGE). We also describe several ways in which the AIOE can be used to create firm level measures of AI exposure. We validate the measures and describe how they can be used in different applications by management, organization and strategy scholars.

Source: https://sms.onlinelibrary.wiley.com/doi/full/10.1002/smj.3286 https://github.com/AIOE-Data https://www.census.gov/topics/employment/industry-occupation/guidance/code-lists.html

## Pipeline:

1. Load industry-level AI exposure (`AIIE`, Appendix B of the AIOE appendix) — keyed by 4-digit NAICS.
2. Clean the 2022 Census→NAICS crosswalk into (Census `IND`, NAICS-pattern) pairs.
3. Build a 2017→2022 `IND` remap from the `2022 Ind Code Changes` sheet so pre-2023 CPS rows can match.
4. For each Census `IND`, find its AIIE — narrow match (AIIE NAICS inside the IND's NAICS scope) with a broader-parent fallback.
5. Left-join onto CPS on the 2022-normalized `IND`.


In [1]:
import re
import pandas as pd

aiie = pd.read_excel("data/AIOE_DataAppendix.xlsx", sheet_name="Appendix B")
aiie.head()

,NAICS,Industry Title,AIIE
0,1133,Logging,-1.360161
1,1151,Support Activities for Crop Production,-2.165304
2,1152,Support Activities for Animal Production,-1.149307
3,2111,Oil and Gas Extraction,0.668615
4,2121,Coal Mining,-1.146242


In [2]:
crosswalk_clean = (
    pd.read_excel(
        "data/2022-Census-Industry-Code-List-with-Crosswalk.xlsx",
        sheet_name="2022 Census Ind Code List ",
    )
    .dropna(how="all")
    .rename(
        columns={
            "Industry 2022 Description": "industry_desc",
            "2022 Census Industry Code": "census_ind_raw",
            "2022 NAICS Code": "naics_raw",
        }
    )
)

is_leaf = ~crosswalk_clean["census_ind_raw"].astype(str).str.contains("-", na=False)
crosswalk_clean = crosswalk_clean.loc[is_leaf].copy()
crosswalk_clean["IND"] = pd.to_numeric(
    crosswalk_clean["census_ind_raw"], errors="coerce"
).astype("Int64")
crosswalk_clean = crosswalk_clean.dropna(subset=["IND"])


def parse_naics_patterns(raw):
    """Return the list of NAICS digit-prefixes encoded in one crosswalk cell."""
    if pd.isna(raw):
        return []
    s = str(raw)
    s = re.sub(
        r"(?i)part of|pt\.?", "", s
    )  # "Part of 311", "pt. 92811" -> "311", "92811"
    s = re.sub(r"(?i)exc\..*", "", s)  # "517 exc. 517111" -> "517"
    patterns = []
    for chunk in s.split(","):
        m = re.match(r"\s*(\d+)", chunk)
        if m:
            patterns.append(m.group(1))
    return patterns


crosswalk_clean["naics_patterns"] = crosswalk_clean["naics_raw"].apply(
    parse_naics_patterns
)
crosswalk_clean[["IND", "industry_desc", "naics_raw", "naics_patterns"]].head(10)

,IND,industry_desc,naics_raw,naics_patterns
4,170,Crop production,111,[111]
5,180,Animal production and aquaculture,112,[112]
6,190,Forestry except logging,"1131, 1132","[1131, 1132]"
7,270,Logging,1133,[1133]
8,280,"Fishing, hunting and trapping",114,[114]
9,290,Support activities for agriculture and forestry,115,[115]
13,370,Oil and gas extraction,211,[211]
14,380,Coal mining,2121,[2121]
15,390,Metal ore mining,2122,[2122]
16,470,Nonmetallic mineral mining and quarrying,2123,[2123]


In [3]:
code_changes = (
    pd.read_excel(
        "data/2022-Census-Industry-Code-List-with-Crosswalk.xlsx",
        sheet_name="2022 Ind Code Changes",
        header=2,
    )
    .dropna(subset=["2017 Census Industry Code"])
    .astype({"2017 Census Industry Code": int, "2022 Census Industry Code": int})
)
ind_2017_to_2022 = dict(
    zip(
        code_changes["2017 Census Industry Code"],
        code_changes["2022 Census Industry Code"],
    )
)
print(f"{len(ind_2017_to_2022)} Census IND codes remapped from 2017 to 2022")
code_changes.head()

38 Census IND codes remapped from 2017 to 2022


,2017 Census Industry Code,2017 Census Industry Description,2022 Census Industry Code,2022 Census Industry Description
0,4680,Other motor vehicle dealers,4681,Other motor vehicle dealers
1,4690,"Automotive parts, accessories, and tire stores",4691,"Automotive parts, accessories, and tire retailers"
2,4770,Furniture and home furnishings stores,4771,Furniture and home furnishings retailers
3,4780,Household appliance stores,4796,Electronics and appliance retailers
4,4795,Electronics stores,4796,Electronics and appliance retailers


In [4]:
# AIIE is published at the 4-digit NAICS level. Census INDs map to NAICS at varying
# granularities, so we do a two-pass match:

aiie_lookup = dict(zip(aiie["NAICS"].astype(str), aiie["AIIE"]))
aiie_keys_longest_first = sorted(aiie_lookup, key=len, reverse=True)


def ind_to_aiie(patterns):
    narrow = [
        aiie_lookup[k]
        for p in patterns
        for k in aiie_keys_longest_first
        if k.startswith(p)
    ]
    if narrow:
        return sum(narrow) / len(narrow)
    for p in patterns:
        for k in aiie_keys_longest_first:  # longest prefix wins
            if p.startswith(k):
                return aiie_lookup[k]
    return None


crosswalk_clean["AIIE"] = crosswalk_clean["naics_patterns"].apply(ind_to_aiie)
ind_aiie = (
    crosswalk_clean.dropna(subset=["AIIE"])
    .astype({"IND": "int64"})[["IND", "industry_desc", "AIIE"]]
    .reset_index(drop=True)
)
print(f"Census INDs with AIIE: {len(ind_aiie)} / {len(crosswalk_clean)}")
ind_aiie.head()

Census INDs with AIIE: 182 / 268


,IND,industry_desc,AIIE
0,270,Logging,-1.360161
1,290,Support activities for agriculture and forestry,-1.657306
2,370,Oil and gas extraction,0.668615
3,380,Coal mining,-1.146242
4,390,Metal ore mining,-0.892403


In [5]:
df = pd.read_csv("data/cps_00001.csv")
df["IND_2022"] = df["IND"].map(lambda x: ind_2017_to_2022.get(x, x))
df_aiie = df.merge(
    ind_aiie.rename(columns={"IND": "IND_2022"}),
    on="IND_2022",
    how="left",
)

employed = df_aiie["IND"] != 0  # IND == 0 is NIU (not employed / not in universe)
n_emp = int(employed.sum())
n_cov = int((employed & df_aiie["AIIE"].notna()).sum())
print(f"CPS rows: {len(df_aiie):,}")
print(f"Employed (IND != 0): {n_emp:,}")
print(f"  with AIIE: {n_cov:,} ({n_cov / n_emp:.1%})")

missing_inds = (
    df_aiie.loc[employed & df_aiie["AIIE"].isna(), "IND"].value_counts().head(10)
)
print("\nTop remaining unmatched CPS INDs:")
missing_inds

CPS rows: 1,528,045
Employed (IND != 0): 775,795
  with AIIE: 599,044 (77.2%)

Top remaining unmatched CPS INDs:


IND
8190    16333
9470    12324
4970     6408
170      6276
9480     6234
9370     6206
6990     6129
7070     6087
4971     5560
9590     5217
Name: count, dtype: int64

# AIOE

Same idea for the occupation side:

1. Load occupation-level AI exposure (`AIOE`, Appendix A) — keyed by SOC code.
2. Clean the 2018 Census Occupation → SOC crosswalk.
3. For each Census `OCC`, find its AIOE — narrow match (AIOE SOCs inside the OCC's SOC scope, averaged) with a broader-parent fallback.
4. Left-join onto the CPS extract on `OCC`.


In [6]:
aioe = pd.read_excel("data/AIOE_DataAppendix.xlsx", sheet_name="Appendix A")
aioe.head()

,SOC Code,Occupation Title,AIOE
0,11-1011,Chief Executives,1.334246
1,11-1021,General and Operations Managers,0.574877
2,11-2011,Advertising and Promotions Managers,1.294387
3,11-2021,Marketing Managers,1.315032
4,11-2022,Sales Managers,1.266280


In [7]:
occ_crosswalk = (
    pd.read_excel(
        "data/2018-occupation-code-list-and-crosswalk.xlsx",
        sheet_name="2018 Census Occ Code List",
        header=0,
    )
    .dropna(how="all")
    .rename(
        columns={
            "2018 Census Title ": "occ_desc",
            "2018 Census Code": "census_occ_raw",
            "2018 SOC Code": "soc_raw",
        }
    )
)

is_leaf = ~occ_crosswalk["census_occ_raw"].astype(str).str.contains("-", na=False)
occ_crosswalk = occ_crosswalk.loc[is_leaf].copy()
occ_crosswalk["OCC"] = pd.to_numeric(
    occ_crosswalk["census_occ_raw"], errors="coerce"
).astype("Int64")
occ_crosswalk = occ_crosswalk.dropna(subset=["OCC"])


def parse_soc_pattern(raw):
    """SOC prefixes up to the first 'X' wildcard. "13-20XX" -> "13-20", "11-1011" -> "11-1011"."""
    if pd.isna(raw):
        return ""
    m = re.match(r"^([\d\-]+)", str(raw).strip())
    return m.group(1) if m else ""


occ_crosswalk["soc_pattern"] = occ_crosswalk["soc_raw"].apply(parse_soc_pattern)
occ_crosswalk[["OCC", "occ_desc", "soc_raw", "soc_pattern"]].head(10)

,OCC,occ_desc,soc_raw,soc_pattern
9,10,Chief executives,11-1011,11-1011
10,20,General and operations managers,11-1021,11-1021
11,30,Legislators,11-1031,11-1031
12,40,Advertising and promotions managers,11-2011,11-2011
13,51,Marketing managers,11-2021,11-2021
14,52,Sales managers,11-2022,11-2022
15,60,Public relations and fundraising managers,11-2030,11-2030
16,101,Administrative services managers,11-3012,11-3012
17,102,Facilities managers,11-3013,11-3013
18,110,Computer and information systems managers,11-3021,11-3021


In [8]:
# AIOE is published at the detailed-SOC level (e.g. "11-1011"). The crosswalk sometimes
# maps a Census OCC to a broad SOC group whose code ends in 0 (e.g. "53-3030"
# Driver/sales workers and truck drivers — the AIOE entries are 53-3031/32/33).
# Matching order: exact narrow match, then broad-group expansion if the pattern
# ends in 0, then a broader-parent fallback.

aioe_lookup = dict(zip(aioe["SOC Code"].astype(str), aioe["AIOE"]))
aioe_keys_longest_first = sorted(aioe_lookup, key=len, reverse=True)


def occ_to_aioe(pattern):
    if not pattern:
        return None
    narrow = [aioe_lookup[k] for k in aioe_keys_longest_first if k.startswith(pattern)]
    if narrow:
        return sum(narrow) / len(narrow)
    if pattern.endswith("0"):
        broad = pattern[:-1]
        group = [aioe_lookup[k] for k in aioe_keys_longest_first if k.startswith(broad)]
        if group:
            return sum(group) / len(group)
    for k in aioe_keys_longest_first:  # longest prefix wins
        if pattern.startswith(k):
            return aioe_lookup[k]
    return None


occ_crosswalk["AIOE"] = occ_crosswalk["soc_pattern"].apply(occ_to_aioe)
occ_aioe = (
    occ_crosswalk.dropna(subset=["AIOE"])
    .astype({"OCC": "int64"})[["OCC", "occ_desc", "AIOE"]]
    .reset_index(drop=True)
)
print(f"Census OCCs with AIOE: {len(occ_aioe)} / {len(occ_crosswalk)}")
occ_aioe.head()

Census OCCs with AIOE: 487 / 570


,OCC,occ_desc,AIOE
0,10,Chief executives,1.334246
1,20,General and operations managers,0.574877
2,40,Advertising and promotions managers,1.294387
3,51,Marketing managers,1.315032
4,52,Sales managers,1.266280


In [9]:
# Add pre-2018 Census OCC codes via the `Occ Code Changes` sheet.
# Each changed 2010 code maps to one or more 2018 codes (splits are common, e.g.
# 0430 "Managers, All Other" split into 0335/0426/0440/0705). When a 2010 code
# targets multiple 2018 codes, we average their AIOE values.

occ_changes = (
    pd.read_excel(
        "data/2018-occupation-code-list-and-crosswalk.xlsx",
        sheet_name="Occ Code Changes",
        header=2,
    )
    .rename(
        columns={
            "2010 Census OCC Code": "occ_2010",
            "2010 Census Title": "title_2010",
            "New 2018 Census OCC Code": "occ_2018",
        }
    )
    .dropna(how="all")
)

# Split rows leave 2010 columns blank — forward-fill the parent 2010 code.
occ_changes[["occ_2010", "title_2010"]] = occ_changes[
    ["occ_2010", "title_2010"]
].ffill()
occ_changes = occ_changes.dropna(subset=["occ_2010", "occ_2018"]).astype(
    {"occ_2010": int, "occ_2018": int}
)

occ_2018_to_aioe = dict(zip(occ_aioe["OCC"], occ_aioe["AIOE"]))
occ_changes["AIOE"] = occ_changes["occ_2018"].map(occ_2018_to_aioe)

pre2018 = (
    occ_changes[~occ_changes["occ_2010"].isin(occ_aioe["OCC"])]
    .dropna(subset=["AIOE"])
    .groupby("occ_2010", as_index=False)
    .agg(occ_desc=("title_2010", "first"), AIOE=("AIOE", "mean"))
    .rename(columns={"occ_2010": "OCC"})
)

occ_aioe_full = pd.concat([occ_aioe, pre2018], ignore_index=True)
print(
    f"OCCs with AIOE after vintage merge: {len(occ_aioe_full)} "
    f"(+{len(pre2018)} pre-2018 codes)"
)
occ_aioe_full.tail()

OCCs with AIOE after vintage merge: 556 (+69 pre-2018 codes)


,OCC,occ_desc,AIOE
551,9520,"Dredge, Excavating, and Loading Machine Operators",-1.237271
552,9560,Hoist and Winch Operators,-1.158975
553,9730,Mine Shuttle Car Operators,-1.182959
554,9740,"Tank Car, Truck, and Ship Loaders",-1.369879
555,9820,Military Enlisted Tactical Operations and Air/...,0.262513


In [10]:
df_aiie = df_aiie.merge(occ_aioe_full, on="OCC", how="left")

working = df_aiie["OCC"] != 0  # OCC == 0 is NIU
n_work = int(working.sum())
n_cov = int((working & df_aiie["AIOE"].notna()).sum())
print(f"Working (OCC != 0): {n_work:,}")
print(f"  with AIOE: {n_cov:,} ({n_cov / n_work:.1%})")

missing_occs = (
    df_aiie.loc[working & df_aiie["AIOE"].isna(), "OCC"].value_counts().head(10)
)
print("\nTop unmatched CPS OCCs:")
missing_occs

Working (OCC != 0): 775,795
  with AIOE: 683,287 (88.1%)

Top unmatched CPS OCCs:


OCC
3600    4853
1021    4237
9645    3902
3602    3642
5620    3578
4610    3552
1020    3262
3603    3179
1050    2713
5940    2691
Name: count, dtype: int64

In [11]:
df_aiie.to_csv("data/data_with_aiie.csv", index=False)